# 06 — Bayesian Regime Model

## 1. Title and objective

**Objective.** Model the equal-weight market return of the seven-stock tech universe as a two-component mixture of low-volatility and high-volatility regimes.

This notebook is intentionally descriptive. It estimates uncertainty around regime-level return and volatility parameters, but it does **not** claim that regimes are directly tradable signals or that the fitted mixture forecasts future market states.

## 2. Why regime modeling is useful

Financial markets do not usually exhibit constant risk through time:

- **Risk changes across market environments.** Calm periods and stressed periods can have materially different return dispersion.
- **Volatility clusters.** Large absolute returns tend to occur near other large absolute returns, while quiet periods often persist.
- **Returns may be generated by multiple states.** A simple descriptive approximation is to treat daily market returns as coming from a mixture of a lower-risk state and a higher-risk state.

The model below is a **two-regime Bayesian Student-t mixture**. The Student-t likelihood gives each regime heavier tails than a Gaussian model. The mixture is simpler than a hidden Markov model because it does not explicitly model transition probabilities or time persistence; it only estimates the probability that an observation was generated by each component, conditional on the fitted component distributions.

## Setup: imports, paths, and reproducibility settings

The path setup below allows the notebook to run from either the repository root or from the `notebooks/` directory. Posterior samples are cached so repeated notebook runs can reuse fitted results unless `REFIT_MODELS` is set to `True`.

In [ ]:
from pathlib import Path
import sys

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.stats import t as student_t

# Resolve the repository root whether this notebook is launched from the repo
# root or from the notebooks/ directory.
NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / "src").exists() else NOTEBOOK_PATH.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.bayesian_models import (
    build_two_regime_market_model,
    sample_model,
    summarize_regime_model,
)
from src.config import (
    DUCKDB_PATH,
    FIGURES_DIR,
    RAW_CSV_PATH,
    STOCK_SYMBOLS,
    TRADING_DAYS_PER_YEAR,
)
from src.model_io import (
    POSTERIOR_SAMPLES_DIR,
    TABLES_DIR,
    ensure_model_dirs,
    load_inference_data,
    save_inference_data,
    save_summary_table,
)
from src.sql_utils import initialize_database, query_to_df, run_sql_pipeline

RANDOM_SEED = 42
DRAWS = 1000
TUNE = 1000
CHAINS = 4
TARGET_ACCEPT = 0.92
ROLLING_WINDOW = 63

# If True, overwrite cached posterior samples. If False, reuse cached NetCDF
# samples when available and refit only when the cache is missing.
REFIT_MODELS = False

RAW_CSV_FOR_NOTEBOOK = RAW_CSV_PATH if RAW_CSV_PATH.exists() else ROOT / "tech_stocks.csv"
POSTERIOR_DIR = POSTERIOR_SAMPLES_DIR
ensure_model_dirs()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
if "arviz-darkgrid" in az.style.available().get("common", []):
    az.style.use("arviz-darkgrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.6f}".format)

rng = np.random.default_rng(RANDOM_SEED)

print(f"Repository root: {ROOT}")
print(f"Raw CSV used:     {RAW_CSV_FOR_NOTEBOOK}")
print(f"DuckDB path:      {DUCKDB_PATH}")
print(f"Figures dir:      {FIGURES_DIR}")
print(f"Posterior dir:    {POSTERIOR_DIR}")

## 3. Load `portfolio_returns_wide` from DuckDB

The SQL pipeline creates a wide daily-return panel with one log-return column per stock. The query below uses the restricted aligned universe, so each row contains observations for all seven selected technology stocks.

In [ ]:
con = initialize_database(raw_csv_path=RAW_CSV_FOR_NOTEBOOK, db_path=DUCKDB_PATH)
run_sql_pipeline(con)

portfolio_returns_wide = query_to_df(
    con,
    """
    SELECT
        date,
        AAPL,
        AMZN,
        FB,
        GOOG,
        MSFT,
        NFLX,
        NVDA
    FROM portfolio_returns_wide
    ORDER BY date
    """,
)
portfolio_returns_wide["date"] = pd.to_datetime(portfolio_returns_wide["date"])
portfolio_returns_wide = portfolio_returns_wide.sort_values("date").reset_index(drop=True)

return_columns = list(STOCK_SYMBOLS)
portfolio_returns_wide[return_columns] = portfolio_returns_wide[return_columns].apply(
    pd.to_numeric, errors="coerce"
)

print(f"Rows loaded: {len(portfolio_returns_wide):,}")
print(f"Date range:  {portfolio_returns_wide['date'].min().date()} to {portfolio_returns_wide['date'].max().date()}")
display(portfolio_returns_wide.head())

## 4. Create equal-weight market return across the seven stocks

The market proxy is the simple equal-weight average of the seven daily log returns. This keeps the regime model focused on broad market-wide movement in the selected tech universe rather than a single company.

## Return convention and tail risk in regimes

The regime model uses aligned **daily log returns** so regime means and volatilities can be annualized consistently: mean daily log return is multiplied by `252`, and daily volatility is multiplied by `sqrt(252)`. A Student-t mixture is used instead of a normal mixture because each regime can still contain unusually large return observations; heavier tails reduce the tendency to force every extreme day into a separate high-volatility state.



In [ ]:
market_returns = portfolio_returns_wide.dropna(subset=return_columns).copy()
market_returns["equal_weight_market_return"] = market_returns[return_columns].mean(axis=1)
market_returns["rolling_63d_volatility"] = (
    market_returns["equal_weight_market_return"]
    .rolling(ROLLING_WINDOW)
    .std()
    * np.sqrt(TRADING_DAYS_PER_YEAR)
)

market_return_array = market_returns["equal_weight_market_return"].to_numpy(dtype="float64")

summary_stats = pd.DataFrame(
    {
        "metric": [
            "observations",
            "mean_daily_return",
            "annualized_mean_return",
            "daily_volatility",
            "annualized_volatility",
            "minimum_daily_return",
            "maximum_daily_return",
        ],
        "value": [
            len(market_returns),
            market_returns["equal_weight_market_return"].mean(),
            market_returns["equal_weight_market_return"].mean() * TRADING_DAYS_PER_YEAR,
            market_returns["equal_weight_market_return"].std(ddof=1),
            market_returns["equal_weight_market_return"].std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR),
            market_returns["equal_weight_market_return"].min(),
            market_returns["equal_weight_market_return"].max(),
        ],
    }
)
display(summary_stats)

## 5. Plot equal-weight market return over time

The daily market-return plot shows when unusually large positive or negative observations occur. These outlying days help motivate a heavy-tailed regime model.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    market_returns["date"],
    market_returns["equal_weight_market_return"],
    color="#4C78A8",
    linewidth=0.8,
)
ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.7)
ax.set_title("Equal-weight daily market log return across seven tech stocks")
ax.set_xlabel("Date")
ax.set_ylabel("Daily log return")
y_abs = market_returns["equal_weight_market_return"].abs().max()
ax.set_ylim(-y_abs * 1.1, y_abs * 1.1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
fig.tight_layout()
market_return_plot_path = FIGURES_DIR / "06_equal_weight_market_return.png"
fig.savefig(market_return_plot_path, dpi=300, bbox_inches="tight")
market_return_plot_path

## 6. Plot rolling 63-day volatility

A 63-trading-day rolling volatility series approximates one quarter of trading days. It is not a Bayesian estimate; it is a descriptive diagnostic showing that volatility tends to cluster through time.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    market_returns["date"],
    market_returns["rolling_63d_volatility"],
    color="#F58518",
    linewidth=1.5,
)
ax.set_title("Rolling 63-day annualized volatility of equal-weight market return")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized volatility")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.tight_layout()
rolling_volatility_plot_path = FIGURES_DIR / "06_rolling_63d_market_volatility.png"
fig.savefig(rolling_volatility_plot_path, dpi=300, bbox_inches="tight")
rolling_volatility_plot_path

## 7. Fit two-regime Bayesian Student-t mixture model

The fitted model has two Student-t components:

- `regime_probs`: mixture probability of each regime.
- `mu_regime`: daily mean return for each regime.
- `sigma_regime`: daily volatility scale for each regime.
- `nu`: shared Student-t degrees-of-freedom parameter.

Mixture models can suffer from label switching, so downstream summaries sort each posterior draw by `sigma_regime`. After sorting, the first component is interpreted as the lower-volatility regime and the second as the higher-volatility regime.

In [ ]:
regime_model_path = POSTERIOR_DIR / "regime_model.nc"

if regime_model_path.exists() and not REFIT_MODELS:
    regime_idata = load_inference_data(regime_model_path)
    print(f"Loaded cached posterior samples from {regime_model_path}")
else:
    regime_model = build_two_regime_market_model(market_return_array)
    regime_idata = sample_model(
        regime_model,
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
    )
    save_inference_data(regime_idata, regime_model_path)
    print(f"Saved posterior samples to {regime_model_path}")

regime_arviz_summary = az.summary(
    regime_idata,
    var_names=["regime_probs", "mu_regime", "sigma_regime", "nu"],
    round_to=6,
)
regime_arviz_summary_path = save_summary_table(
    regime_arviz_summary.reset_index(names="parameter"),
    TABLES_DIR / "06_regime_model_arviz_summary.csv",
)
print(f"Saved regime model ArviZ summary to {regime_arviz_summary_path}")
regime_arviz_summary


## 8. Summarize posterior regime parameters

The summary below reports posterior regime probability, daily mean, annualized mean, daily volatility, and annualized volatility. Credible intervals are intentionally shown to avoid over-interpreting point estimates.

In [ ]:
regime_summary = summarize_regime_model(regime_idata)

regime_parameter_summary = regime_summary[
    [
        "regime_label",
        "regime_probability_mean",
        "regime_probability_q5",
        "regime_probability_q50",
        "regime_probability_q95",
        "mean_return_mean",
        "mean_return_q5",
        "mean_return_q50",
        "mean_return_q95",
        "volatility_mean",
        "volatility_q5",
        "volatility_q50",
        "volatility_q95",
    ]
].copy()
regime_parameter_summary["annualized_mean_mean"] = (
    regime_parameter_summary["mean_return_mean"] * TRADING_DAYS_PER_YEAR
)
regime_parameter_summary["annualized_mean_q5"] = (
    regime_parameter_summary["mean_return_q5"] * TRADING_DAYS_PER_YEAR
)
regime_parameter_summary["annualized_mean_q50"] = (
    regime_parameter_summary["mean_return_q50"] * TRADING_DAYS_PER_YEAR
)
regime_parameter_summary["annualized_mean_q95"] = (
    regime_parameter_summary["mean_return_q95"] * TRADING_DAYS_PER_YEAR
)
regime_parameter_summary["annualized_volatility_mean"] = (
    regime_parameter_summary["volatility_mean"] * np.sqrt(TRADING_DAYS_PER_YEAR)
)
regime_parameter_summary["annualized_volatility_q5"] = (
    regime_parameter_summary["volatility_q5"] * np.sqrt(TRADING_DAYS_PER_YEAR)
)
regime_parameter_summary["annualized_volatility_q50"] = (
    regime_parameter_summary["volatility_q50"] * np.sqrt(TRADING_DAYS_PER_YEAR)
)
regime_parameter_summary["annualized_volatility_q95"] = (
    regime_parameter_summary["volatility_q95"] * np.sqrt(TRADING_DAYS_PER_YEAR)
)

ordered_columns = [
    "regime_label",
    "regime_probability_mean",
    "regime_probability_q5",
    "regime_probability_q50",
    "regime_probability_q95",
    "mean_return_mean",
    "mean_return_q5",
    "mean_return_q50",
    "mean_return_q95",
    "annualized_mean_mean",
    "annualized_mean_q5",
    "annualized_mean_q50",
    "annualized_mean_q95",
    "volatility_mean",
    "volatility_q5",
    "volatility_q50",
    "volatility_q95",
    "annualized_volatility_mean",
    "annualized_volatility_q5",
    "annualized_volatility_q50",
    "annualized_volatility_q95",
]
regime_parameter_summary = regime_parameter_summary[ordered_columns]

regime_parameter_summary_path = save_summary_table(
    regime_parameter_summary,
    TABLES_DIR / "06_regime_parameter_summary.csv",
)
print(f"Saved regime parameter summary to {regime_parameter_summary_path}")

display(
    regime_parameter_summary.style.format(
        {
            "regime_probability_mean": "{:.1%}",
            "regime_probability_q5": "{:.1%}",
            "regime_probability_q50": "{:.1%}",
            "regime_probability_q95": "{:.1%}",
            "mean_return_mean": "{:.3%}",
            "mean_return_q5": "{:.3%}",
            "mean_return_q50": "{:.3%}",
            "mean_return_q95": "{:.3%}",
            "annualized_mean_mean": "{:.1%}",
            "annualized_mean_q5": "{:.1%}",
            "annualized_mean_q50": "{:.1%}",
            "annualized_mean_q95": "{:.1%}",
            "volatility_mean": "{:.3%}",
            "volatility_q5": "{:.3%}",
            "volatility_q50": "{:.3%}",
            "volatility_q95": "{:.3%}",
            "annualized_volatility_mean": "{:.1%}",
            "annualized_volatility_q5": "{:.1%}",
            "annualized_volatility_q50": "{:.1%}",
            "annualized_volatility_q95": "{:.1%}",
        }
    )
)

## 9. Visualize posterior regime parameters

The next charts compare posterior uncertainty in the low- and high-volatility components. Since the labels are based on sorted posterior volatility, they should be interpreted as relative component descriptions rather than hard economic states.

In [ ]:
def sorted_regime_posterior_samples(idata):
    """Return posterior samples sorted by volatility within each draw."""
    posterior = idata.posterior.stack(sample=("chain", "draw"))
    mu = posterior["mu_regime"].transpose("regime", "sample").values
    sigma = posterior["sigma_regime"].transpose("regime", "sample").values
    probs = posterior["regime_probs"].transpose("regime", "sample").values
    nu = posterior["nu"].values.reshape(-1)

    order = np.argsort(sigma, axis=0)
    sorted_mu = np.take_along_axis(mu, order, axis=0)
    sorted_sigma = np.take_along_axis(sigma, order, axis=0)
    sorted_probs = np.take_along_axis(probs, order, axis=0)

    return {
        "mu": sorted_mu,
        "sigma": sorted_sigma,
        "probability": sorted_probs,
        "nu": nu,
    }

sorted_samples = sorted_regime_posterior_samples(regime_idata)
annualized_sigma_samples = sorted_samples["sigma"] * np.sqrt(TRADING_DAYS_PER_YEAR)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    annualized_sigma_samples[0],
    bins=40,
    stat="density",
    kde=True,
    color="#4C78A8",
    ax=ax,
)
ax.set_title("Posterior distribution of low-regime annualized volatility")
ax.set_xlabel("Annualized volatility")
ax.set_ylabel("Posterior density")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.tight_layout()
low_volatility_posterior_path = FIGURES_DIR / "06_low_regime_volatility_posterior.png"
fig.savefig(low_volatility_posterior_path, dpi=300, bbox_inches="tight")
low_volatility_posterior_path

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    annualized_sigma_samples[1],
    bins=40,
    stat="density",
    kde=True,
    color="#E45756",
    ax=ax,
)
ax.set_title("Posterior distribution of high-regime annualized volatility")
ax.set_xlabel("Annualized volatility")
ax.set_ylabel("Posterior density")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.tight_layout()
high_volatility_posterior_path = FIGURES_DIR / "06_high_regime_volatility_posterior.png"
fig.savefig(high_volatility_posterior_path, dpi=300, bbox_inches="tight")
high_volatility_posterior_path

In [ ]:
probability_plot_df = pd.DataFrame(
    {
        "regime": ["Low volatility", "High volatility"],
        "mean": regime_parameter_summary["regime_probability_mean"],
        "q5": regime_parameter_summary["regime_probability_q5"],
        "q95": regime_parameter_summary["regime_probability_q95"],
    }
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    probability_plot_df["mean"],
    probability_plot_df["regime"],
    xerr=[
        probability_plot_df["mean"] - probability_plot_df["q5"],
        probability_plot_df["q95"] - probability_plot_df["mean"],
    ],
    fmt="o",
    color="#72B7B2",
    ecolor="#72B7B2",
    elinewidth=2,
    capsize=4,
)
ax.set_xlim(0, 1)
ax.set_title("Posterior regime probability comparison")
ax.set_xlabel("Mixture probability")
ax.set_ylabel("Regime")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.tight_layout()
regime_probability_path = FIGURES_DIR / "06_regime_probability_comparison.png"
fig.savefig(regime_probability_path, dpi=300, bbox_inches="tight")
regime_probability_path

## 10. Approximate high-volatility responsibility by day

For each day, the approximate high-volatility responsibility is computed from the posterior mixture parameters and the observed return. For every posterior draw, the two components are sorted by volatility and Bayes' rule is applied to estimate the conditional probability that the observed return came from the high-volatility component. The final value is the posterior average of that day-level probability.

This is an approximation from a static mixture model. It should not be read as a filtered hidden-state probability from a hidden Markov model.

In [ ]:
def estimate_high_volatility_responsibility(returns, sorted_samples, batch_size=250):
    """Estimate day-level posterior responsibility for the high-volatility component."""
    returns = np.asarray(returns, dtype="float64")
    n_samples = sorted_samples["sigma"].shape[1]
    high_prob_sum = np.zeros(returns.size, dtype="float64")

    for start in range(0, n_samples, batch_size):
        stop = min(start + batch_size, n_samples)
        sample_slice = slice(start, stop)

        low_density = student_t.pdf(
            returns[:, None],
            df=sorted_samples["nu"][sample_slice][None, :],
            loc=sorted_samples["mu"][0, sample_slice][None, :],
            scale=sorted_samples["sigma"][0, sample_slice][None, :],
        )
        high_density = student_t.pdf(
            returns[:, None],
            df=sorted_samples["nu"][sample_slice][None, :],
            loc=sorted_samples["mu"][1, sample_slice][None, :],
            scale=sorted_samples["sigma"][1, sample_slice][None, :],
        )
        low_weighted = sorted_samples["probability"][0, sample_slice][None, :] * low_density
        high_weighted = sorted_samples["probability"][1, sample_slice][None, :] * high_density
        denominator = low_weighted + high_weighted
        high_probability = np.divide(
            high_weighted,
            denominator,
            out=np.zeros_like(high_weighted),
            where=denominator > 0,
        )
        high_prob_sum += high_probability.sum(axis=1)

    return high_prob_sum / n_samples

market_returns["high_volatility_responsibility"] = estimate_high_volatility_responsibility(
    market_return_array,
    sorted_samples,
)

responsibility_summary = market_returns[
    ["date", "equal_weight_market_return", "rolling_63d_volatility", "high_volatility_responsibility"]
].copy()

responsibility_summary.sort_values(
    "high_volatility_responsibility", ascending=False
).head(10)

## 11. Overlay high-volatility periods on market return and rolling volatility

The shaded points below use a transparent red color proportional to the estimated high-volatility responsibility. Darker points indicate days that the static mixture model associates more strongly with the higher-volatility component.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    market_returns["date"],
    market_returns["equal_weight_market_return"],
    color="#4C78A8",
    linewidth=0.7,
    alpha=0.75,
    label="Equal-weight market return",
)
scatter = ax.scatter(
    market_returns["date"],
    market_returns["equal_weight_market_return"],
    c=market_returns["high_volatility_responsibility"],
    cmap="Reds",
    s=18,
    alpha=0.85,
    label="High-volatility responsibility",
)
ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.7)
ax.set_title("Market returns colored by approximate high-volatility responsibility")
ax.set_xlabel("Date")
ax.set_ylabel("Daily log return")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label("High-volatility responsibility")
fig.tight_layout()
high_volatility_return_overlay_path = FIGURES_DIR / "06_high_volatility_responsibility_market_return_overlay.png"
fig.savefig(high_volatility_return_overlay_path, dpi=300, bbox_inches="tight")
high_volatility_return_overlay_path

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    market_returns["date"],
    market_returns["rolling_63d_volatility"],
    color="#F58518",
    linewidth=1.5,
    label="63-day annualized volatility",
)
scatter = ax.scatter(
    market_returns["date"],
    market_returns["rolling_63d_volatility"],
    c=market_returns["high_volatility_responsibility"],
    cmap="Reds",
    s=18,
    alpha=0.85,
)
ax.set_title("Rolling volatility colored by approximate high-volatility responsibility")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized volatility")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label("High-volatility responsibility")
fig.tight_layout()
high_volatility_rolling_overlay_path = FIGURES_DIR / "06_high_volatility_responsibility_rolling_volatility_overlay.png"
fig.savefig(high_volatility_rolling_overlay_path, dpi=300, bbox_inches="tight")
high_volatility_rolling_overlay_path

## 12. Interpretation and limitations

- The high-volatility regime should generally show a larger posterior volatility estimate than the low-volatility regime. It may also have wider uncertainty around mean returns because high-volatility observations are noisier.
- Days with large negative or positive moves can receive higher high-volatility responsibility. In historical market data, large drawdowns often appear during high-volatility periods, but the model is not designed to explain every drawdown or predict the next one.
- This model is **descriptive, not a trading signal**. It summarizes historical return behavior and uncertainty in a compact way.
- A two-component mixture is simpler than a full hidden Markov model. It does not estimate persistence, transition probabilities, or a latent time path with state dynamics.
- Results depend on the selected seven-stock universe, equal weighting, date range, priors, and the static mixture assumption. Treat the output as an uncertainty-aware summary rather than a definitive classification of market regimes.

## 13. Saved figure files

The notebook saves all generated figures to `reports/figures` for use in reports or dashboards.

In [ ]:
saved_figure_paths = [
    market_return_plot_path,
    rolling_volatility_plot_path,
    low_volatility_posterior_path,
    high_volatility_posterior_path,
    regime_probability_path,
    high_volatility_return_overlay_path,
    high_volatility_rolling_overlay_path,
]

for figure_path in saved_figure_paths:
    print(figure_path.relative_to(ROOT))